# Store Sales データセットのカラム説明

エクアドルの小売チェーン **Corporación Favorita** の時系列販売予測 (Kaggle: Store Sales — Time Series Forecasting) のデータセット。
期間は概ね 2013-01-01 〜 2017-08-31。train は 2013-01-01〜2017-08-15、test は 2017-08-16〜2017-08-31。

## train.csv (3,000,888 行)

| カラム | 型 | 説明 |
|---|---|---|
| `id` | int | 行を一意に識別する連番 |
| `date` | date | 日付 (YYYY-MM-DD) |
| `store_nbr` | int | 店舗番号 (1〜54) |
| `family` | category | 商品カテゴリ (33 種類、例: AUTOMOTIVE, BABY CARE, BEVERAGES, BOOKS …) |
| `sales` | float | その店舗・カテゴリ・日付の売上（予測ターゲット） |
| `onpromotion` | int | その店舗でその日プロモーション対象となっているそのカテゴリの商品数 (0〜303) |

1 行が「1 店舗 × 1 カテゴリ × 1 日」の組み合わせに対応。

## test.csv (28,512 行)

| カラム | 型 | 説明 |
|---|---|---|
| `id` | int | 提出用の行識別子（sample_submission の id と一致） |
| `date` | date | 日付 (2017-08-16 〜 2017-08-31) |
| `store_nbr` | int | 店舗番号 |
| `family` | category | 商品カテゴリ |
| `onpromotion` | int | プロモーション対象商品数 |

`sales` が無く、これを予測する。

## sample_submission.csv (28,512 行)

| カラム | 型 | 説明 |
|---|---|---|
| `id` | int | test.csv の id |
| `sales` | float | 予測売上（提出時はこの列に値を入れる） |

## stores.csv (54 行)

| カラム | 型 | 説明 |
|---|---|---|
| `store_nbr` | int | 店舗番号 |
| `city` | category | 店舗所在地の都市 (23 種類) |
| `state` | category | 州 (17 種類) |
| `type` | category | 店舗タイプ (A〜E の 5 種類) |
| `cluster` | int | 店舗クラスタ (1〜17)。類似店舗のグループ分け |

## holidays_events.csv (350 行)

| カラム | 型 | 説明 |
|---|---|---|
| `date` | date | 祝日・イベントの日付 (2012-03-02 〜 2017-08-15) |
| `type` | category | 種類: `Holiday`, `Additional`, `Bridge`, `Transfer`, `Work Day`, `Event` |
| `locale` | category | 適用範囲: `National`, `Local`, `Regional` |
| `locale_name` | category | locale に対応する国・地域・都市の名前 |
| `description` | category | 祝日・イベントの説明（例: Black Friday, Carnival） |
| `transferred` | bool | 祝日が別の日に振り替えられたか (`True`/`False`)。振り替え時は元の日付にも転送済みとして残る |

## oil.csv (1,218 行)

| カラム | 型 | 説明 |
|---|---|---|
| `date` | date | 日付 (2013-01-01 〜 2017-08-15) |
| `dcoilwtico` | float | 日次 WTI 原油価格 (米ドル/バレル)。欠損あり |

## transactions.csv (83,488 行)

| カラム | 型 | 説明 |
|---|---|---|
| `date` | date | 日付 (2013-01-01 〜 2017-08-15) |
| `store_nbr` | int | 店舗番号 |
| `transactions` | int | その店舗のその日の取引（レジ通過）総数 |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 全セルで共通利用するデータを1回だけ読み込む（再実行の高速化）。
# 各セルが個別に read_csv するのをやめ、このロードセルが唯一の読み込み元にする。
train = pd.read_csv('../data/train.csv', usecols=['date', 'store_nbr', 'family', 'sales'], parse_dates=['date'])
tx = pd.read_csv('../data/transactions.csv')
stores = pd.read_csv('../data/stores.csv')
he = pd.read_csv('../data/holidays_events.csv', parse_dates=['date'])


In [ ]:

train = pd.read_csv('../data/train.csv', usecols=['store_nbr', 'sales'])

# 店舗ごとの総行数は全て同一 (33カテゴリ × 1684日 = 55572)。
# 店舗間の差は「売上>0 の有効データ量」に出る。
per_store = train.groupby('store_nbr')['sales'].size().rename('total_rows')
n_sold = train[train['sales'] > 0].groupby('store_nbr').size().rename('rows_with_sales')
zero_ratio = (train.groupby('store_nbr')['sales'].apply(lambda s: (s == 0).mean())).rename('zero_ratio')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].bar(n_sold.index, n_sold.values, color='steelblue')
axes[0].axhline(per_store.mean(), color='crimson', ls='--', lw=1, label='total rows (55572)')
axes[0].set_xlabel('store_nbr')
axes[0].set_ylabel('rows with sales > 0')
axes[0].set_title('Valid (sales>0) rows per store')
axes[0].legend()
axes[0].grid(axis='y', linestyle='--', alpha=0.4)

axes[1].bar(zero_ratio.index, zero_ratio.values, color='salmon')
axes[1].set_xlabel('store_nbr')
axes[1].set_ylabel('fraction of sales == 0')
axes[1].set_title('Zero-sales ratio per store')
axes[1].grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()


In [ ]:

# 日次・店舗単位の総売上
train = pd.read_csv('../data/train.csv', usecols=['date', 'store_nbr', 'sales'])
daily_sales = train.groupby(['date', 'store_nbr'], as_index=False)['sales'].sum()

# 取引数と結合して sales per transaction を算出
tx = tx
m = daily_sales.merge(tx, on=['date', 'store_nbr'], how='inner')
m['sales_per_transaction'] = m['sales'] / m['transactions']

med = m.groupby('store_nbr')['sales_per_transaction'].median()
q1 = m.groupby('store_nbr')['sales_per_transaction'].quantile(0.25)
q3 = m.groupby('store_nbr')['sales_per_transaction'].quantile(0.75)
# 棒は中央値、エラーバーは中央値±(Q1, Q3) の四分位範囲 (IQR)。
# spt は右に歪む分布で平均±std は外れ値に引きずられる。店舗2 のように平均が Q3 を
# 超える店もあるため、平均ではなく中央値を中心にして常に非負のエラーバーにする。
err_lower = med - q1
err_upper = q3 - med

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(med.index, med.values, yerr=np.stack([err_lower.values, err_upper.values]), capsize=3, color='seagreen', alpha=0.85, label='median ± IQR')
ax.axhline(m['sales_per_transaction'].median(), color='crimson', ls='--', lw=1, label='overall median')
ax.set_xlabel('store_nbr')
ax.set_ylabel('median sales per transaction')
ax.set_title('Median sales per transaction by store (IQR error bars)')

ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:

# 店舗日次の spt（cell2 と同じ計算）
train = pd.read_csv('../data/train.csv', usecols=['date', 'store_nbr', 'sales'])
daily = train.groupby(['date', 'store_nbr'], as_index=False)['sales'].sum()
tx = tx
m = daily.merge(tx, on=['date', 'store_nbr'])
m['spt'] = m['sales'] / m['transactions']
store_spt = m.groupby('store_nbr')['spt'].mean().rename('spt').reset_index()

stores = stores
s = store_spt.merge(stores, on='store_nbr')

# 属性別の平均 spt（店舗数が少ない属性は箱ひげ図だと薄いので平均棒で）
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

by_type = s.groupby('type')['spt'].mean()
axes[0,0].bar(by_type.index, by_type.values, color='steelblue')
axes[0,0].set_title('avg spt by type')
axes[0,0].set_xlabel('type')
axes[0,0].grid(axis='y', ls='--', alpha=0.4)

by_cluster = s.groupby('cluster')['spt'].mean().sort_values()
axes[0,1].bar(by_cluster.index.astype(str), by_cluster.values, color='seagreen')
axes[0,1].set_title('avg spt by cluster')
axes[0,1].set_xlabel('cluster')
axes[0,1].tick_params(axis='x', rotation=90)
axes[0,1].grid(axis='y', ls='--', alpha=0.4)

by_state = s.groupby('state')['spt'].mean().sort_values()
axes[1,0].barh(by_state.index, by_state.values, color='darkorange')
axes[1,0].set_title('avg spt by state')
axes[1,0].set_xlabel('avg spt')
axes[1,0].grid(axis='x', ls='--', alpha=0.4)

by_city = s.groupby('city')['spt'].mean().sort_values()
axes[1,1].barh(by_city.index, by_city.values, color='rebeccapurple')
axes[1,1].set_title('avg spt by city')
axes[1,1].set_xlabel('avg spt')
axes[1,1].grid(axis='x', ls='--', alpha=0.4)

plt.tight_layout()
plt.show()

# family は取引数が存在しないため spt を直接計算できない。
# 参考として family 別の平均売上（店舗・日次あたり）を示す。
train_f = train[["store_nbr", "family", "sales"]]
fam_sales = train_f.groupby('family')['sales'].mean().sort_values()
fig2, ax2 = plt.subplots(figsize=(10, 10))
ax2.barh(fam_sales.index, fam_sales.values, color='teal')
ax2.set_title('avg daily sales by family')
ax2.set_xlabel('avg sales')
ax2.grid(axis='x', ls='--', alpha=0.4)
plt.tight_layout()
plt.show()


## 店舗ごとの平均売上 vs. sales per transaction の考察

- **平均売上（店舗単位の売上総量）** が高い店舗は必ずしも **客単価（sales per transaction）** が高いわけではない。
- 例: 平均売上トップの **店舗 44**（平均売上 1117.25）は spt が 8.37 と中位。つまり **取引数（客数）が多く**、客単価は平均的。
- 逆に客単価が突出して高いのは **店舗 51（11.43）・42（11.05）・21（10.90）**。これらは取引数が平均的〜少なめ（51: 1714, 42: 1115, 21: 1127）なのに、1回の買い物金額が大きい（高額商品・大口購入の多い店）。
- 客単価が低い店舗は **34（4.44）・14（5.02）・15（5.18）・12（5.41）**。
- 全体平均は **7.46**、中央値 7.16。

**予測への示唆:** 「売上 = 客単価 × 取引数」なので、店舗ごとにどちらの要因で売上が決まるかが異なる。プロモーション（onpromotion）や曜日・祝日の影響は、客単価重視の店と取引数重視の店で効き方が変わると考えられ、店舗（または cluster）ごとにモデルを分ける判断材料になる。

In [ ]:

# 日次・店舗単位の総売上（family を合計）
train = pd.read_csv('../data/train.csv', usecols=['date', 'store_nbr', 'sales'], parse_dates=['date'])
ds = train.groupby(['date', 'store_nbr'], as_index=False)['sales'].sum()

# 店ごとの平均日次売上（基準値）
store_mean = ds.groupby('store_nbr')['sales'].mean().rename('store_mean')

# 各店のその日の売上を、その店自身の平均売上に対する比率に正規化。
# 店の規模差を除いた「相対的な売れ具合」を店間で比較できる。
ds['ratio'] = ds['sales'] / ds['store_nbr'].map(store_mean)

# 日毎の全体売上（全店合計）
daily_total = ds.groupby('date')['sales'].sum()

# 日毎の店舗間の比率のばらつき（中央値 ± IQR）
disp = ds.groupby('date')['ratio'].agg(
    med='median',
    q1=lambda s: s.quantile(0.25),
    q3=lambda s: s.quantile(0.75),
)

fig, axes = plt.subplots(
    2, 1, figsize=(16, 9), sharex=True,
    gridspec_kw={'height_ratios': [2, 1]},
)

# 上段: 日毎の全体売上
axes[0].plot(daily_total.index, daily_total.values, color='steelblue', lw=1)
axes[0].set_ylabel('total sales (all stores)')
axes[0].set_title('Daily total sales across all stores')
axes[0].grid(axis='y', linestyle='--', alpha=0.4)

# 下段: 店舗間の分散を「各店の平均売上に対する比率」で表現。
# 中心は日毎の店舗中央値（通常 ≈1、祝日など全体が伸びる日は >1）、
# エラーバーは店舗間の四分位範囲 (IQR)。sales は右に歪むため平均±std ではなく中央値±IQR を使う。
err_lower = disp['med'] - disp['q1']
err_upper = disp['q3'] - disp['med']
axes[1].errorbar(
    disp.index, disp['med'].values,
    yerr=np.stack([err_lower.values, err_upper.values]),
    fmt='o', ms=2, capsize=0, lw=0.6, alpha=0.35,
    color='seagreen', ecolor='seagreen', zorder=2,
)
axes[1].axhline(1.0, color='crimson', ls='--', lw=1, label='baseline (each store = its own mean)')
axes[1].set_ylabel('sales / store mean')
axes[1].set_title('Store dispersion: daily sales relative to each store mean (median ± IQR)')
axes[1].legend()
axes[1].grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()


In [ ]:

# 日毎・店舗単位の総売上（前セルと同じ計算）
train = pd.read_csv('../data/train.csv', usecols=['date', 'store_nbr', 'sales'], parse_dates=['date'])
ds = train.groupby(['date', 'store_nbr'], as_index=False)['sales'].sum()
daily_total = ds.groupby('date')['sales'].sum()

# ISO 週（月曜始まり）にまとめる
df = daily_total.to_frame('total')
df['week'] = df.index.to_period('W')

# 各日の売上を「その週の平均」に対する比率に正規化。
# 週ごとに平均=1 に揃うので、中心（中央値）は ≈1 になり、幅が週内の相対分散を表す。
wm = df.groupby('week')['total'].transform('mean')
df['ratio'] = df['total'] / wm

# 週ごとの分散（中央値 ± IQR）
disp = df.groupby('week')['ratio'].agg(
    med='median',
    q1=lambda s: s.quantile(0.25),
    q3=lambda s: s.quantile(0.75),
)
week_total = df.groupby('week')['total'].sum()
x = week_total.index.to_timestamp()

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True, gridspec_kw={'height_ratios': [1, 1]})

# 上段: 週ごとの総売上（レベル）
axes[0].plot(x, week_total.values, color='steelblue', lw=1)
axes[0].set_ylabel('weekly total sales')
axes[0].set_title('Weekly total sales')
axes[0].grid(axis='y', ls='--', alpha=0.4)

# 下段: 週平均に対する日次売上の分散（中央値 ± IQR）
err_lower = disp['med'] - disp['q1']
err_upper = disp['q3'] - disp['med']
axes[1].errorbar(
    x, disp['med'].values,
    yerr=np.stack([err_lower.values, err_upper.values]),
    fmt='o', ms=3, capsize=0, lw=0.6, alpha=0.4,
    color='seagreen', ecolor='seagreen', zorder=2,
)
axes[1].axhline(1.0, color='crimson', ls='--', lw=1, label='= week mean')
axes[1].set_ylabel('daily / week mean')
axes[1].set_title('Intra-week dispersion: daily sales relative to the week mean (median ± IQR)')
axes[1].legend()
axes[1].grid(axis='y', ls='--', alpha=0.4)

plt.tight_layout()
plt.show()


In [ ]:

# 日毎・店舗単位の総売上
train = pd.read_csv('../data/train.csv', usecols=['date', 'store_nbr', 'sales'], parse_dates=['date'])
ds = train.groupby(['date', 'store_nbr'], as_index=False)['sales'].sum()
daily_total = ds.groupby('date')['sales'].sum()

# 曜日ごとの平均売上と分散（std）。
# 日次合計は曜日内では概ね対称分布（平均≈中央値）なので、平均 ± std を使う。
df = daily_total.to_frame('total')
df['dow'] = df.index.dayofweek  # Mon=0 .. Sun=6
g = df.groupby('dow')['total']
mean = g.mean()
std = g.std()

labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
x = np.arange(7)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x, mean.values, yerr=std.values, capsize=4, color='steelblue', alpha=0.85, label='mean ± std')
ax.axhline(daily_total.mean(), color='crimson', ls='--', lw=1, label='overall daily mean')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('daily total sales')
ax.set_title('Average daily sales by weekday (error bar = std across weeks)')
ax.legend()
ax.grid(axis='y', ls='--', alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:

# 日毎・店舗単位の総売上
train = pd.read_csv('../data/train.csv', usecols=['date', 'store_nbr', 'sales'], parse_dates=['date'])
ds = train.groupby(['date', 'store_nbr'], as_index=False)['sales'].sum()
daily_total = ds.groupby('date')['sales'].sum()

# 月にまとめる（週セルと同じ構造: 上=レベル, 下=月平均に対する日次分散）
df = daily_total.to_frame('total')
df['mon'] = df.index.to_period('M')
wm = df.groupby('mon')['total'].transform('mean')
df['ratio'] = df['total'] / wm

disp = df.groupby('mon')['ratio'].agg(
    med='median',
    q1=lambda s: s.quantile(0.25),
    q3=lambda s: s.quantile(0.75),
)
month_total = df.groupby('mon')['total'].sum()
x = month_total.index.to_timestamp()

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True, gridspec_kw={'height_ratios': [1, 1]})

axes[0].plot(x, month_total.values, color='steelblue', lw=1)
axes[0].set_ylabel('monthly total sales')
axes[0].set_title('Monthly total sales')
axes[0].grid(axis='y', ls='--', alpha=0.4)

err_lower = disp['med'] - disp['q1']
err_upper = disp['q3'] - disp['med']
axes[1].errorbar(
    x, disp['med'].values,
    yerr=np.stack([err_lower.values, err_upper.values]),
    fmt='o', ms=3, capsize=0, lw=0.6, alpha=0.4,
    color='seagreen', ecolor='seagreen', zorder=2,
)
axes[1].axhline(1.0, color='crimson', ls='--', lw=1, label='= month mean')
axes[1].set_ylabel('daily / month mean')
axes[1].set_title('Intra-month dispersion: daily sales relative to the month mean (median ± IQR)')
axes[1].legend()
axes[1].grid(axis='y', ls='--', alpha=0.4)

plt.tight_layout()
plt.show()


In [ ]:

# 日毎・店舗単位の総売上
train = pd.read_csv('../data/train.csv', usecols=['date', 'store_nbr', 'sales'], parse_dates=['date'])
ds = train.groupby(['date', 'store_nbr'], as_index=False)['sales'].sum()
daily_total = ds.groupby('date')['sales'].sum()

df = daily_total.to_frame('total')
df['dow'] = df.index.dayofweek  # Mon=0 .. Sun=6
week_mean = df.groupby('dow')['total'].mean()

# 曜日で正規化: 各日の売上を「その曜日の平均」で割る。
# これで曜日による高低（ジグザグの主因）を除いた残差が見える。
df['ratio'] = df['total'] / df['dow'].map(week_mean)

labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
colors = ['#4e79a7', '#f28e2b', '#59a14f', '#e15759', '#b07aa1', '#76b7b2', '#edc949']
dowc = [colors[i] for i in df['dow'].values]

fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=True, gridspec_kw={'height_ratios': [2, 1]})

# 上段: 日毎売上を曜日で色分け。破線 = 各曜日の平均（比較基準）
axes[0].scatter(df.index, df['total'].values, c=dowc, s=4, alpha=0.5)
for d in range(7):
    axes[0].axhline(week_mean[d], color=colors[d], ls='--', lw=1, alpha=0.7)
axes[0].set_ylabel('daily total sales')
axes[0].set_title('Daily total sales colored by weekday (dashed = weekday mean)')
axes[0].grid(axis='y', ls='--', alpha=0.4)
handles = [mpatches.Patch(color=colors[i], label=labels[i]) for i in range(7)]
axes[0].legend(handles=handles, ncol=7, loc='upper right', framealpha=0.8)

# 下段: 曜日で正規化した比率（売上 / その曜日の平均）。1.0 が基準。
# ジグザグ（曜日効果）を除くと、残るのは祝日スパイクや長期的な傾向。
axes[1].scatter(df.index, df['ratio'].values, c=dowc, s=4, alpha=0.5)
axes[1].axhline(1.0, color='k', lw=1)
axes[1].set_ylabel('sales / weekday mean')
axes[1].set_title('Weekday-de-seasonalized: daily sales / its weekday mean (weekday zigzag removed)')
axes[1].grid(axis='y', ls='--', alpha=0.4)

plt.tight_layout()
plt.show()


In [ ]:

# 日毎・店舗単位の総売上
train = pd.read_csv('../data/train.csv', usecols=['date', 'store_nbr', 'sales'], parse_dates=['date'])
ds = train.groupby(['date', 'store_nbr'], as_index=False)['sales'].sum()
daily_total = ds.groupby('date')['sales'].sum()

# 曜日で正規化した比率
df = daily_total.to_frame('total')
df['dow'] = df.index.dayofweek
week_mean = df.groupby('dow')['total'].mean()
df['ratio'] = df['total'] / df['dow'].map(week_mean)

# x軸は日次のまま、分散は週次(7日ローリング窓)で算出して各日にエラーバーを付ける
df['rmean'] = df['ratio'].rolling(7, center=True, min_periods=1).mean()
df['rstd']  = df['ratio'].rolling(7, center=True, min_periods=1).std()

fig, ax = plt.subplots(figsize=(16, 5))
# 生の日次比率（薄いグレー線）
ax.plot(df.index, df['ratio'].values, color='lightgray', lw=0.6, alpha=0.6, label='daily ratio', zorder=1)
# 各日に7日窓の分散をエラーバーで付ける（日次解像度）
ax.errorbar(df.index, df['rmean'].values, yerr=df['rstd'].values,
            fmt='none', capsize=0, lw=0.4, alpha=0.5, color='steelblue', ecolor='steelblue', zorder=2)
ax.plot(df.index, df['rmean'].values, color='steelblue', lw=0.8, zorder=3, label='7-day rolling mean ± std')
ax.axhline(1.0, color='crimson', ls='--', lw=1, label='baseline (weekday mean)')
ax.set_ylabel('sales / weekday mean')
ax.set_xlabel('date')
ax.set_title('Daily de-seasonalized ratio with weekly (7-day rolling) dispersion error bars')
ax.legend()
ax.grid(axis='y', ls='--', alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:

# 日毎の曜日正規化比率（前セルと同じ計算）
train = pd.read_csv('../data/train.csv', usecols=['date', 'store_nbr', 'sales'], parse_dates=['date'])
ds = train.groupby(['date', 'store_nbr'], as_index=False)['sales'].sum()
daily_total = ds.groupby('date')['sales'].sum()
df = daily_total.to_frame('total')
df['dow'] = df.index.dayofweek
week_mean = df.groupby('dow')['total'].mean()
df['ratio'] = df['total'] / df['dow'].map(week_mean)
x = df['ratio'].values - 1.0  # 中心化（平均 0）
days = (df.index - df.index.min()).days.values


def top_periods(signal, max_period=120):
    """FFT パワースペクトルの上位周期（日）。ハニング窓でスペクトル漏れを軽減。"""
    n = len(signal)
    freq = np.fft.rfftfreq(n, d=1)  # 周期/日
    spec = np.abs(np.fft.rfft(signal * np.hanning(n)))**2
    mask = (freq > 1 / max_period) & (freq > 0)
    order = np.argsort(spec[mask])[::-1]
    return freq[mask], spec[mask], order


# --- 1) FFT ピリオドグラム（全体） ---
freq, spec, order = top_periods(x)
print('FFT top periods (full period):')
for i in order[:5]:
    print(f'  {1/freq[i]:6.2f} days  power {spec[i]:.0f}')

# --- 2) 年ごとの上位周期 ---
print('\nper-year top periods:')
for year, sub in df.groupby(df.index.year):
    xx = sub['ratio'].values - 1.0
    f, s, o = top_periods(xx)
    tops = ', '.join(f'{1/f[i]:.1f}d' for i in o[:3])
    print(f'  {year}: {tops}')

# --- 3) 折り返しスキャン: 周期 P を 2〜120 日で全探索 ---
# 各 P でデータを P 日ごとに折り返し、グループ間分散の割合 (eta^2) が最大の P が
# その周期の「当てはまりの良さ」。※注意: 非整数周期 (例 15.17日) は整数 P で折り返すと
# 位相が徐々にズレるため、P=15 より P=61 (=15.2x4) や P=91 (=15.2x6) のような
# 倍数が上位に来る罠がある。基本周期の特定は FFT / 自己相関の方が正確。
res = []
for P in range(2, 121):
    phase = days % P
    g = pd.Series(x).groupby(phase)
    eta2 = (g.size() * g.mean()**2).sum() / (x**2).sum()
    res.append((eta2, P))
best_eta, best_P = sorted(res, reverse=True)[0]
print('\nfold-scan best:', f'P={best_P} days, eta^2={best_eta:.3f}')

# --- 4) 自己相関（ラグ=日） ---
n = len(x)
ac = np.correlate(x, x, 'full')[n-1:n+120]
ac = ac / ac[0]
lags = np.arange(121)
print('\nautocorrelation top lags:')
pk = [(l, ac[l]) for l in range(3, 118)
      if ac[l] > ac[l-1] and ac[l] >= ac[l+1] and ac[l] > 0.3]
for l, v in pk[:5]:
    print(f'  lag {l:3d} days  r={v:.3f}')

# --- 可視化 ---
fig, axes = plt.subplots(3, 1, figsize=(15, 10))

axes[0].plot(1/freq, spec, color='steelblue')
for i in order[:4]:
    pd_, s_ = 1/freq[i], spec[i]
    axes[0].axvline(pd_, color='crimson', ls='--', lw=0.8)
    axes[0].annotate(f'{pd_:.1f}d', (pd_, s_), textcoords='offset points', xytext=(0, 6), ha='center', color='crimson')
axes[0].set_xlim(2, 120)
axes[0].set_xlabel('period (days)')
axes[0].set_ylabel('FFT power')
axes[0].set_title('FFT periodogram (weekday-de-seasonalized ratio)')
axes[0].grid(alpha=0.3)

axes[1].plot([p for _, p in res], [e for e, _ in res], color='seagreen')
axes[1].axvline(best_P, color='crimson', ls='--', lw=0.8)
axes[1].annotate(f'P={best_P}d', (best_P, best_eta), textcoords='offset points', xytext=(0, 6), ha='center', color='crimson')
axes[1].set_xlabel('fold period P (days)')
axes[1].set_ylabel('explained variance eta^2')
axes[1].set_title('Fold-scan: how well period P explains the wave (multiples win — see comment)')
axes[1].grid(alpha=0.3)

axes[2].plot(lags, ac, color='rebeccapurple')
for l, v in pk[:4]:
    axes[2].axvline(l, color='crimson', ls='--', lw=0.8)
    axes[2].annotate(f'{l}d', (l, v), textcoords='offset points', xytext=(0, 6), ha='center', color='crimson')
axes[2].set_xlabel('lag (days)')
axes[2].set_ylabel('autocorrelation')
axes[2].set_title('Autocorrelation by lag (days)')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 周期検出の説明（上のコードセルがやっていること）

### 背景
「曜日（月〜日）の平均」で正規化して曜日効果を取り除いたのに、残った波（起伏）はまだ周期的に見える。
このコードセルは **その波の周期が何日なのか** を、4 つのやり方で調べている。

### 1. FFT（フーリエ変換）ピリオドグラム
波を「いろいろな周期のサイン波」に分解し、**どの周期がどれだけ強く含まれるか** を測る。
横軸 = 周期（日）、縦軸 = パワー（強さ）。パワーが大きい周期ほど、その波が強い。

- 結果: **15.17 日**が最強（次いで 30.1 日）。
  → 「約 15 日ごとに山が来る波」が支配的だということ。

### 2. 年ごとの上位周期
同じ分析を年ごとに分けて実行し、「周期が年によって変わるか」を確認。

- 結果: 2013・2015・2016・2017 は 15 日系。**2014 だけ 60.7 日・91 日系が上位**。
  → 2014 年はリズムが違う可能性（要調査）。

### 3. 折り返しスキャン（周期を日で全探索）
「周期 P 日ごとにデータを折り返して重ねたとき、山と谷がどれだけ揃うか」を、P = 2〜120 日で全部試す。
揃いの良さを eta²（説明分散比）で測り、最大の P が「波を一番よく説明する周期」。

- 結果: **P = 91 日**が最適。
- **罠**: 非整数の周期（例 15.17 日）を整数 P で折り返すと、1 周期ごとに位相が少しずつズレる。
  P = 15 だとズレが蓄積して波が潰れるが、P = 61（= 15.2 × 4）や P = 91（= 15.2 × 6）は
  ほぼ「ちょうど整数周期分」の折り返しになるので揃う。**だから倍数が上位に来てしまう**。
  基本周期を出すなら FFT の方が正確。

### 4. 自己相関
「元の波と、d 日だけずらした波の似ている度合い」を、ずらし量（ラグ）d ごとに計算。
d が周期の倍数に近いと高くなる。

- 結果: ラグ 14, 30, 44, 61 日でピーク。**ピークの間隔が約 15 日** → 15 日周期の裏付け。

### 結論
- FFT の基本周期 ≈ 15.2 日、自己相関のピーク間隔 ≈ 15 日 → **約 15 日の周期がある**。
- 「約 15 日」は**月 2 回のリズム**（例: 15 日と月末）と整合的。
  エクアドルでは給料日が月 2 回（15 日と月末）のケースが多く、給料日に買い物が増えるという仮説が立つ。
- 確定するには、**day-of-month（1〜31 日）で平均を取る**のが次の手。


In [ ]:

# 日毎の曜日正規化比率（前セルと同じ）
train = pd.read_csv('../data/train.csv', usecols=['date', 'store_nbr', 'sales'], parse_dates=['date'])
ds = train.groupby(['date', 'store_nbr'], as_index=False)['sales'].sum()
daily_total = ds.groupby('date')['sales'].sum()
df = daily_total.to_frame('total')
df['dow'] = df.index.dayofweek
week_mean = df.groupby('dow')['total'].mean()
df['ratio'] = df['total'] / df['dow'].map(week_mean)
df['dom'] = df.index.day  # 月の何日目か（1〜31）

# day-of-month ごとの平均比率（1.0 = その日の曜日平均と等しい）
g = df.groupby('dom')['ratio']
mean = g.mean()
n = g.size()

fig, ax = plt.subplots(figsize=(13, 5))
# 月初（1〜5日）と月末（29〜31日）を赤で強調
colors = ['crimson' if (d <= 5 or d >= 29) else 'steelblue' for d in mean.index]
ax.bar(mean.index, mean.values, color=colors, alpha=0.85)
ax.axhline(1.0, color='k', ls='--', lw=1, label='baseline (= weekday mean)')
# 29〜31日は短い月にしか存在しないため n が少ない。注記。
for d in [29, 30, 31]:
    ax.annotate(f'n={n[d]}', (d, mean[d]), textcoords='offset points', xytext=(0, 6), ha='center', fontsize=9)
ax.set_xlabel('day of month')
ax.set_ylabel('mean sales / weekday mean')
ax.set_title('Average sales by day-of-month (relative to each weekday mean)')
ax.legend()
ax.grid(axis='y', ls='--', alpha=0.4)
plt.tight_layout()
plt.show()

print('peak (top 5):', ', '.join(f'{d}日={mean[d]:.2f}' for d in mean.sort_values(ascending=False).head(5).index))
print('trough (bottom 5):', ', '.join(f'{d}日={mean[d]:.2f}' for d in mean.sort_values().head(5).index))


## day-of-month 分析（月の日付で売上を比較）

### これは何をしているか
FFT で「周期 ≈ 15 日」という結果が出た。その**正体が月の何日に現れているか**を直接確認する分析。
- 各日を「その月の何日目か（1〜31）」で分類する。
- その日の売上が「その曜日の平均」に対して何倍かを平均する（`ratio`）。
  - ratio > 1 → その日は曜日平均より売れている
  - ratio < 1 → その日は曜日平均より売れていない

### 読み方
- 横軸 = 日付（1日〜31日）、縦軸 = 平均 ratio。
- 1.0 の黒破線が「曜日平均と等しい」基準。
- 赤い棒 = 月初（1〜5日）と月末（29〜31日）。青い棒 = それ以外。

### 結果
- **月初（1〜5日）が最も高い**（ratio 1.06〜1.18）。トップは 2日（1.18）・3日（1.14）・1日（1.12）。
- **中旬（13〜15日）が最も低い**（0.94 前後）。
- **月末（30〜31日）にかけて再び上昇**（1.02〜1.09）。
- つまり「**月の初めに買い物が集中し、中旬に落ちて月末に戻る**」という月周期。
  → 面白いことに **15日そのものは山ではなく谷**。FFT の「15日周期」は、月初と月末の複数の山が分解された見え方だった可能性が高い。

### 注意
- 29〜31日は**短い月にしか存在しない**ためサンプル数が少ない（31日は n=32）。図に n を注記。
- これは平均なので外れ値（祝日）の影響を受けうる。必要なら中央値・IQR で見直せる。


In [ ]:

# イベント（祝日・記念日）データを読み込む
he = he

# 年を無視して「月-日」を 365 日の何日目か（1〜365）に変換（うるう年を考慮しない固定マッピング）
cum = [0, 31, 59, 90, 120, 151, 181, 212, 243, 273, 304, 334]
def md_to_doy(m, d):
    return cum[m - 1] + d
he['doy'] = [md_to_doy(m, d) for m, d in zip(he['date'].dt.month, he['date'].dt.day)]

all_cnt = he.groupby('doy').size().reindex(range(1, 366), fill_value=0)
nat_cnt = he[he['locale'] == 'National'].groupby('doy').size().reindex(range(1, 366), fill_value=0)
local_cnt = all_cnt - nat_cnt  # 国全体に影響しない Local / Regional

doy = np.arange(1, 366)

fig, axes = plt.subplots(2, 1, figsize=(16, 9), gridspec_kw={'height_ratios': [2, 1]})

# 上段: 365日ごとのイベント数（National を赤、それ以外を青の積み上げ）
axes[0].bar(doy, local_cnt.values, color='steelblue', label='Local/Regional', width=1.0)
axes[0].bar(doy, nat_cnt.values, bottom=local_cnt.values, color='crimson', label='National', width=1.0)
for c in cum[1:]:
    axes[0].axvline(c, color='gray', lw=0.5, alpha=0.5)
axes[0].set_xticks(cum)
axes[0].set_xticklabels([f'{i}月' for i in range(1, 13)])
axes[0].set_xlim(0, 365)
axes[0].set_ylabel('event count (all years)')
axes[0].set_title('Event count by day-of-year (365 days), national vs local')
axes[0].legend()
axes[0].grid(axis='y', ls='--', alpha=0.3)
top = nat_cnt.sort_values(ascending=False).head(8)
for d, v in top.items():
    axes[0].annotate(f'{d}', (d, v), textcoords='offset points', xytext=(0, 6), ha='center', fontsize=8, color='crimson')

# 下段: National イベントの day-of-month 分布（売上チャートと直接比較できる）
nat_dom = he[he['locale'] == 'National'].groupby(he['date'].dt.day).size().reindex(range(1, 32), fill_value=0)
axes[1].bar(nat_dom.index, nat_dom.values, color='crimson', alpha=0.85)
axes[1].set_xlabel('day of month')
axes[1].set_ylabel('national event count')
axes[1].set_title('National event count by day-of-month (1-31)')
axes[1].grid(axis='y', ls='--', alpha=0.3)

plt.tight_layout()
plt.show()


## イベントの日付分布（365日）

### これは何をしているか
「day-of-month の売上パターンは、**イベント（祝日・記念日）が多い日に引っ張られている**」という仮説を確認する。
- 祝日データ（`holidays_events.csv`）を読み、年を無視して**月-日 → 365日の何日目か**に変換。
- 日ごとのイベント数を積み上げ棒で表示。**National（国全体に影響）を赤、Local/Regional（一部店舗のみ）を青**。
  → 全国の日次売上を動かすのは National だけ。Local は特定都市の記念日（例: 06-25 は Machala/Latacunga/Imbabura の創設記念が毎年）で、全体売上にはほぼ影響しない。

### 上段の読み方
- 横軸 = 1月1日からの通算日数（1〜365）、月の境界をグレー線で表示。
- 棒の高さ = その日に重なるイベント数（複数年を合算）。
- 大きな赤い山は National イベントの集中（12/24〜26, 05/01, 10/09, 11/02〜03, 08/10, 12/31 など）。赤の数字は通算日。

### 下段の読み方（売上チャートと直接比較）
- National イベントの day-of-month（1〜31日）分布。
- **月初（1〜5日）と月末（24〜31日）に多く、中旬（13〜19日）は疎**。
  → これは day-of-month 売上（月初ピーク・中旬谷）と**おおまかに一致**。イベント密度が売上パターンに影響している可能性を示唆。

### 注意
- これは「イベントがある日」の数であって、イベントの強さ（祝日かどうか）は見ていない。National でも休日（閉店）とイベント（客増）では逆に作用する。
- 次の検証候補: イベントの無い日だけの day-of-month 売上と比べる、または type（Holiday vs Event）で分ける。


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 日毎の曜日正規化比率
train = pd.read_csv('../data/train.csv', usecols=['date', 'store_nbr', 'sales'], parse_dates=['date'])
ds = train.groupby(['date', 'store_nbr'], as_index=False)['sales'].sum()
daily_total = ds.groupby('date')['sales'].sum()
df = daily_total.to_frame('total')
df['dow'] = df.index.dayofweek
week_mean = df.groupby('dow')['total'].mean()
df['ratio'] = df['total'] / df['dow'].map(week_mean)
df['dom'] = df.index.day
df['year'] = df.index.year

# (年, 日付) ごとの平均 ratio をまず計算し、日付ごとに「年をまたいだ」平均と std を取る
y = df.groupby(['year', 'dom'])['ratio'].mean().reset_index()
g = y.groupby('dom')['ratio']
mean = g.mean()
std = g.std()
x = mean.index

fig, ax = plt.subplots(figsize=(13, 5))
colors = ['crimson' if (d <= 5 or d >= 29) else 'steelblue' for d in x]
ax.errorbar(x, mean.values, yerr=std.values, fmt='o', ms=4, capsize=2, lw=0.8,
            color='steelblue', ecolor='gray', alpha=0.85, zorder=2, label='mean ± year-std')
ax.scatter(x, mean.values, c=colors, zorder=3, s=30)
ax.axhline(1.0, color='k', ls='--', lw=1, label='baseline (= weekday mean)')
ax.set_xlabel('day of month')
ax.set_ylabel('mean sales / weekday mean')
ax.set_title('Day-of-month average sales, error bar = year-to-year std (dispersion)')
ax.legend()
ax.grid(axis='y', ls='--', alpha=0.4)
plt.tight_layout()
plt.show()

print('year-std by range:')
print(f'  start-of-month (1-5): mean {mean[1:6].mean():.2f}, year-std {std[1:6].mean():.2f}')
print(f'  mid-month (13-19):    mean {mean[13:20].mean():.2f}, year-std {std[13:20].mean():.2f}')
print(f'  end-of-month (29-31): mean {mean[29:32].mean():.2f}, year-std {std[29:32].mean():.2f}')


## day-of-month パターンの年ごとの分散（エラーバー）

### これは何をしているか
先ほどの day-of-month 平均は**全年を混ぜた値**だった。それだと「特定の年のイベント」に引っ張られて山が出来てる可能性がある。
このセルでは **各日付（1〜31日）について、年ごとの平均 ratio をまず計算し、その年のばらつき（std）をエラーバー**で表示する。

- 中心 = その日付の平均 ratio（年をまたぐ）
- エラーバー = 年ごとの平均のばらつき（**年によってどれだけ変わるか**）
- エラーバーが大きい日 = **毎年同じとは限らず、特定の年のイベントに依存**

### 読み方
- 横軸 = 日付（1〜31日）、縦軸 = ratio。
- 点が 1.0 の黒破線からどれだけ離れているか = 売上の偏り。
- 赤点 = 月初（1〜5日）と月末（29〜31日）。

### 結果
- **月初（1〜5日）は平均が高い（1.08〜1.21）が、年ごとの std も最大（0.30〜0.36）**。
  → エラーバーが大きい。**「月初の売上ピーク」は毎年安定ではなく、特定の年にイベントが重なって出来ている**可能性が高い。
- **中旬（13〜15日）は平均が低く（0.94〜0.96）、年-std もやや小さい（0.26〜0.28）** → 比較的安定して低い。
- 月末（29〜31日）は平均が高いが年-std も大きい（0.27〜0.32）。

### 結論
- 年-std（≈0.26〜0.36）はどこでも 1.0 からのズレ（≈0.1〜0.2）より大きい。
  → day-of-month の「月初ピーク・中旬谷」は **毎年の安定した月周期というより、特定の年のイベント密度に依存して揺れている** と読める。
- 次の検証: イベントの無い年だけに絞って day-of-month 平均を見れば、イベント抜きの純粋な月周期が確認できる。
